---

# Unit 4 - Assignment: Evaluated Agentic RAG System

## Overview

This assignment builds a **self-evaluating agentic RAG system**: a CrewAI pipeline where:

1. **Agent 1 (RAG Retriever)** searches a FAISS vector store and generates an initial answer
2. **Agent 2 (Quality Evaluator)** scores the answer with DeepEval Faithfulness and Answer Relevancy metrics
3. **Agent 3 (Revisor)** activates only on FAIL — rewrites the answer grounded in the retrieved context

```
USER QUESTION
      │
      ▼
┌─────────────────────────────────────────────────────┐
│  AGENT 1: RAG Retriever                             │
│  Queries FAISS vector store → generates answer      │
│  Outputs: (answer, retrieved_context)               │
└────────────────────────┬────────────────────────────┘
                         ▼
┌─────────────────────────────────────────────────────┐
│  AGENT 2: Quality Evaluator                         │
│  FaithfulnessMetric + AnswerRelevancyMetric         │
│  Outputs: scores, PASS/FAIL, reasons                │
└────────────────────────┬────────────────────────────┘
                         │
              ┌──────────┴──────────┐
           PASS                   FAIL
              │                     ▼
         Final Answer     ┌─────────────────────────┐
                          │  AGENT 3: Revisor        │
                          │  Re-generates answer     │
                          │  grounded in context     │
                          └────────────┬────────────┘
                                       ▼
                                 Final Answer (revised)
```

---

## Setup

In [1]:
%pip install -q crewai crewai-tools langchain langchain-groq langchain-community langchain-text-splitters faiss-cpu sentence-transformers deepeval python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API Key: ")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# DeepEval uses an OpenAI-compatible interface — point it at Groq
os.environ["OPENAI_API_KEY"]  = GROQ_API_KEY
os.environ["OPENAI_API_BASE"] = "https://api.groq.com/openai/v1"

print(f"GROQ_API_KEY: {'set' if GROQ_API_KEY else 'NOT SET — add to .env or enter above'}")

GROQ_API_KEY: set


---

## Part 1 - Knowledge Base

**Topic: Space Exploration and Astronomy**

This topic provides rich, factual content with clear answers — ideal for testing RAG faithfulness and retrieval quality. It spans distinct sub-topics (planets, missions, telescopes, physics) so the retriever must work hard to find the right chunk for each question.

The knowledge base contains 600+ words covering the Solar System, Mars exploration, the James Webb Space Telescope, black holes, and the history of space travel.

In [3]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

KNOWLEDGE_BASE = """
The Solar System consists of the Sun and all objects gravitationally bound to it, including eight planets,
five officially recognised dwarf planets, numerous moons, asteroids, and comets. The eight planets in order
from the Sun are Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptune. Jupiter is the largest
planet, with a mass more than twice that of all other planets combined. Saturn is famous for its extensive
ring system, which is composed primarily of ice particles and rocky debris.

Mars is the fourth planet from the Sun and is often called the Red Planet due to the iron oxide on its surface.
NASA's Perseverance rover landed on Mars on 18 February 2021 in Jezero Crater, a location believed to be an
ancient lake bed. Perseverance carries the Ingenuity helicopter, the first powered aircraft to fly on another
planet. The primary mission of Perseverance is to search for signs of ancient microbial life and collect rock
samples for potential return to Earth. Mars has two small moons, Phobos and Deimos, which are thought to be
captured asteroids.

The James Webb Space Telescope (JWST) launched on 25 December 2021 and reached its operational orbit at the
second Lagrange point (L2), approximately 1.5 million kilometres from Earth. JWST observes primarily in the
infrared spectrum, allowing it to see through dust clouds and observe the earliest galaxies formed after the
Big Bang. Its primary mirror is 6.5 metres in diameter, composed of 18 hexagonal gold-coated beryllium
segments. JWST is a collaboration between NASA, the European Space Agency (ESA), and the Canadian Space Agency.
The telescope is named after James E. Webb, who led NASA during the Apollo era.

A black hole is a region of spacetime where gravity is so strong that nothing, not even light, can escape
once it crosses the event horizon. The boundary of a black hole is called the event horizon. Stellar black
holes form when a massive star collapses under its own gravity at the end of its life. The supermassive black
hole at the centre of the Milky Way galaxy is called Sagittarius A* and has a mass approximately four million
times that of the Sun. In 2019, the Event Horizon Telescope collaboration released the first image of a black
hole, located in the galaxy Messier 87.

The first human spaceflight was achieved by Soviet cosmonaut Yuri Gagarin on 12 April 1961, who orbited Earth
once aboard the Vostok 1 spacecraft. The Apollo programme, run by NASA, successfully landed humans on the Moon
six times between 1969 and 1972. Neil Armstrong and Buzz Aldrin became the first humans to walk on the Moon on
20 July 1969 during the Apollo 11 mission. The International Space Station (ISS) has been continuously
inhabited since November 2000 and orbits Earth at approximately 400 kilometres altitude, completing about
16 orbits per day. The ISS is a joint project involving five space agencies: NASA, Roscosmos, JAXA, ESA,
and the Canadian Space Agency.

The Hubble Space Telescope was launched in 1990 and remains one of the most important astronomical instruments
in history. Despite an initial flaw in its primary mirror, it was repaired by Space Shuttle astronauts in 1993
and has since provided high-resolution images of distant galaxies, nebulae, and star clusters. Hubble orbits
Earth at approximately 547 kilometres altitude and operates in ultraviolet, visible, and near-infrared light.
The telescope is named after astronomer Edwin Hubble, who demonstrated in the 1920s that galaxies beyond the
Milky Way exist and that the universe is expanding.
"""

# Split into overlapping chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs     = splitter.create_documents([KNOWLEDGE_BASE])

print(f"Split into {len(docs)} chunks.")
for i, d in enumerate(docs):
    print(f"  Chunk {i}: {len(d.page_content.split())} words — {d.page_content[:80]}...")

Split into 17 chunks.
  Chunk 0: 32 words — The Solar System consists of the Sun and all objects gravitationally bound to it...
  Chunk 1: 48 words — from the Sun are Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, and Neptu...
  Chunk 2: 42 words — Mars is the fourth planet from the Sun and is often called the Red Planet due to...
  Chunk 3: 34 words — ancient lake bed. Perseverance carries the Ingenuity helicopter, the first power...
  Chunk 4: 21 words — samples for potential return to Earth. Mars has two small moons, Phobos and Deim...
  Chunk 5: 33 words — The James Webb Space Telescope (JWST) launched on 25 December 2021 and reached i...
  Chunk 6: 33 words — infrared spectrum, allowing it to see through dust clouds and observe the earlie...
  Chunk 7: 32 words — segments. JWST is a collaboration between NASA, the European Space Agency (ESA),...
  Chunk 8: 39 words — A black hole is a region of spacetime where gravity is so strong that nothing, n...
  Chunk 9: 40 words — ho

In [4]:
print("Building FAISS vector store with sentence-transformers embeddings...")
embeddings   = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore  = FAISS.from_documents(docs, embeddings)
retriever    = vectorstore.as_retriever(search_kwargs={"k": 3})

# Quick sanity check
test_docs = retriever.invoke("Who was the first human in space?")
print(f"\nRetriever sanity check (3 docs retrieved):")
for d in test_docs:
    print(f"  → {d.page_content[:100]}...")
print("\nVector store ready.")

Building FAISS vector store with sentence-transformers embeddings...


C:\Users\diyab\AppData\Local\Temp\ipykernel_27360\3780327980.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings   = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



Retriever sanity check (3 docs retrieved):
  → The first human spaceflight was achieved by Soviet cosmonaut Yuri Gagarin on 12 April 1961, who orbi...
  → six times between 1969 and 1972. Neil Armstrong and Buzz Aldrin became the first humans to walk on t...
  → times that of the Sun. In 2019, the Event Horizon Telescope collaboration released the first image o...

Vector store ready.


---

## Part 2 - RAG Agent

The RAG agent uses a `@tool`-decorated function to query the FAISS vector store.
Its task output includes **both the answer and the retrieved context** — the evaluator needs the context to check faithfulness.

In [49]:
from deepeval.models import DeepEvalBaseLLM
from langchain_groq import ChatGroq

class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.client = ChatGroq(
            model="llama-3.1-8b-instant",
            temperature=0,
            groq_api_key=GROQ_API_KEY
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        return self.client.invoke(prompt).content  

    async def a_generate(self, prompt: str) -> str:
        result = await self.client.ainvoke(prompt)
        return result.content 

    def get_model_name(self) -> str:
        return "llama-3.1-8b-instant"

judge_llm = GroqJudge()
print("GroqJudge ready for DeepEval.")

GroqJudge ready for DeepEval.


In [50]:
import os
import time
import json
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

crew_llm = LLM(
    model="groq/llama-3.1-8b-instant", 
    temperature=0.1,
    max_tokens=300,
    api_key=GROQ_API_KEY
)

# ── RAG tool: queries the FAISS vector store ──────────────────────────────────
@tool("RAG Search Tool")
def rag_search(query: str) -> str:
    """
    Search the knowledge base vector store for relevant context.
    Use this tool to retrieve factual information before answering any question.

    Args:
        query (str): The question or search query.

    Returns:
        str: Retrieved context passages separated by newlines.
    """
    retrieved = retriever.invoke(query)
    context_parts = []
    for i, doc in enumerate(retrieved, 1):
        context_parts.append(f"[Context {i}]\n{doc.page_content.strip()}")
    return "\n\n".join(context_parts)

print("RAG tool and LLM configured.")

RAG tool and LLM configured.


In [51]:
rag_agent = Agent(
    role="RAG Retriever",
    goal=(
        "Search the knowledge base for relevant context and produce a factual answer. "
        "Your output MUST follow this exact format:\n"
        "ANSWER: <your answer here>\n"
        "CONTEXT: <the retrieved context passages you used>"
    ),
    backstory=(
        "You are a precise research assistant that only answers from retrieved evidence. "
        "You never hallucinate and always cite the context you used."
    ),
    tools=[rag_search],
    allow_delegation=False,
    verbose=False,
    llm=crew_llm
)

print("RAG agent defined.")

RAG agent defined.


In [22]:
def run_rag_task(question: str) -> str:
    """Run the RAG agent on a single question and return its output string."""
    task = Task(
        description=(
            f"Question: {question}\n\n"
            "Step 1: Use the RAG Search Tool with this question.\n"
            "Step 2: Write a concise factual answer (2-4 sentences) based ONLY on retrieved context.\n"
            "Step 3: Output in this EXACT format:\n"
            "ANSWER: <answer>\n"
            "CONTEXT: <the context passages you retrieved>"
        ),
        agent=rag_agent,
        expected_output="ANSWER: ... CONTEXT: ..."
    )
    crew   = Crew(agents=[rag_agent], tasks=[task], verbose=False)
    result = crew.kickoff()
    return str(result)

# Sample output for 3 test questions
sample_questions = [
    "When did the Perseverance rover land on Mars?",
    "What is the James Webb Space Telescope and where does it orbit?",
    "Who was the first human to travel to space?"
]

print("Running RAG agent on 3 sample questions...")
print("=" * 70)
for q in sample_questions:
    print(f"\nQ: {q}")
    out = run_rag_task(q)
    print(out[:400])
    print("-" * 70)
    time.sleep(3)

Running RAG agent on 3 sample questions...

Q: When did the Perseverance rover land on Mars?


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

ANSWER: The Perseverance rover landed on Mars on 18 February 2021.
CONTEXT: Mars is the fourth planet from the Sun and is often called the Red Planet due to the iron oxide on its surface.
NASA's Perseverance rover landed on Mars on 18 February 2021 in Jezero Crater, a location believed to be an
ancient lake bed. Perseverance carries the Ingenuity helicopter, the first powered aircraft to fly on an
----------------------------------------------------------------------

Q: What is the James Webb Space Telescope and where does it orbit?


...

ANSWER: The James Webb Space Telescope (JWST) is a space telescope that orbits at the second Lagrange point (L2), approximately 1.5 million kilometres from Earth. JWST is a collaboration between NASA, the European Space Agency (ESA), and the Canadian Space Agency, and it observes primarily in the infrared segment. The telescope is named after James E. Webb, who led NASA during the Apollo era.
CONT
----------------------------------------------------------------------

Q: Who was the first human to travel to space?


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

ANSWER: The first human to travel to space was Soviet cosmonaut Yuri Gagarin, who orbited Earth once aboard the Vostok 1 spacecraft on 12 April 1961. 
CONTEXT: The first human spaceflight was achieved by Soviet cosmonaut Yuri Gagarin on 12 April 1961, who orbited Earth once aboard the Vostok 1 spacecraft. The Apollo programme, run by NASA, successfully landed humans on the Moon six times between 1
----------------------------------------------------------------------


---

## Part 3 - Quality Evaluator Agent

The evaluator agent wraps **DeepEval** `FaithfulnessMetric` and `AnswerRelevancyMetric` inside a `@tool`.
Both metrics use `model="llama-3.1-8b-instant"` as the judge LLM.

- **Faithfulness** checks: is every claim in the answer supported by the retrieved context?
- **Answer Relevancy** checks: does the answer actually address the question asked?
- Threshold for both: **0.7** — below this is FAIL

In [29]:
from deepeval.test_case import LLMTestCase
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

@tool("DeepEval Quality Checker")
def deepeval_quality_check(question: str, answer: str, context: str) -> str:
    """
    Evaluate an answer using DeepEval Faithfulness and Answer Relevancy metrics.
    Returns a JSON-formatted evaluation result with scores, pass/fail, and reasons.

    Args:
        question (str): The original user question.
        answer (str):   The answer generated by the RAG agent.
        context (str):  The retrieved context passages used to generate the answer.

    Returns:
        str: JSON string with faithfulness_score, relevancy_score, verdict, reasons.
    """
    context_list = [c.strip() for c in context.split("[Context") if c.strip()]
    if not context_list:
        context_list = [context]

    tc = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=context_list
    )

    faithfulness_metric  = FaithfulnessMetric(threshold=0.7, model=judge_llm, include_reason=True)
    relevancy_metric     = AnswerRelevancyMetric(threshold=0.7, model=judge_llm, include_reason=True)

    faithfulness_metric.measure(tc)
    relevancy_metric.measure(tc)

    f_score  = faithfulness_metric.score  if faithfulness_metric.score  is not None else 0.0
    r_score  = relevancy_metric.score     if relevancy_metric.score     is not None else 0.0
    f_reason = faithfulness_metric.reason or "No reason provided."
    r_reason = relevancy_metric.reason    or "No reason provided."

    verdict  = "PASS" if (f_score >= 0.7 and r_score >= 0.7) else "FAIL"
    reasons  = []
    if f_score < 0.7:
        reasons.append(f"Faithfulness below threshold: {f_reason}")
    if r_score < 0.7:
        reasons.append(f"Relevancy below threshold: {r_reason}")

    result = {
        "faithfulness_score": round(f_score, 3),
        "relevancy_score":    round(r_score, 3),
        "verdict":            verdict,
        "reasons":            reasons if reasons else ["All metrics passed threshold."]
    }
    return json.dumps(result)


evaluator_agent = Agent(
    role="Quality Evaluator",
    goal=(
        "Evaluate the RAG answer using the DeepEval Quality Checker tool. "
        "Extract the ANSWER and CONTEXT from the RAG output, then call the tool. "
        "Report the verdict clearly: PASS or FAIL with specific reasons."
    ),
    backstory=(
        "You are a strict LLM quality auditor. You use quantitative metrics to assess "
        "whether answers are faithful to their source and relevant to the question."
    ),
    tools=[deepeval_quality_check],
    allow_delegation=False,
    verbose=False,
    llm=crew_llm
)

print("Evaluator agent and DeepEval tool defined.")

Evaluator agent and DeepEval tool defined.


In [30]:
import time

# Sample evaluation on one question
sample_q   = "When did the Perseverance rover land on Mars?"
sample_rag = run_rag_task(sample_q)

# Extract answer and context from RAG output
rag_lines = sample_rag.strip().split("\n")
sample_answer  = ""
sample_context = ""
for line in rag_lines:
    if line.startswith("ANSWER:"):
        sample_answer = line.replace("ANSWER:", "").strip()
    elif line.startswith("CONTEXT:"):
        sample_context = line.replace("CONTEXT:", "").strip()

if not sample_context:
    sample_context = sample_rag[:1000]

# trim to avoid rate limits
sample_answer  = sample_answer[:300]
sample_context = sample_context[:1000]

eval_task = Task(
    description=(
        f"Question: {sample_q}\n\n"
        f"RAG Answer: {sample_answer}\n\n"
        f"Retrieved Context: {sample_context}\n\n"
        "Evaluate the answer based on faithfulness and relevancy.\n"
        "Return ONLY JSON in this format:\n"
        "{\n"
        "  \"faithfulness_score\": float,\n"
        "  \"relevancy_score\": float,\n"
        "  \"verdict\": \"GOOD\" or \"BAD\",\n"
        "  \"reasons\": \"brief explanation\"\n"
        "}"
    ),
    agent=evaluator_agent,
    expected_output="JSON evaluation result"
)

eval_crew = Crew(
    agents=[evaluator_agent],
    tasks=[eval_task],
    verbose=False,
    memory=False
)

try:
    eval_result = eval_crew.kickoff()
except Exception:
    time.sleep(5)
    eval_result = eval_crew.kickoff()

print("Sample Evaluation Output:")
print("=" * 60)
print(str(eval_result))

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

Output()

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

Sample Evaluation Output:
{
  "faithfulness_score": 0.95,
  "relevancy_score": 0.9,
  "verdict": "GOOD",
  "reasons": "The answer is faithful to the source as it provides the correct date of the Perseverance rover's landing on Mars. The answer is also relevant to the question as it directly answers the query."
}


---

## Part 4 - Revisor Agent

The revisor agent activates **only when the evaluator returns FAIL**.
It receives:
- The original question
- The failed answer
- The evaluator's specific failure reasons
- The retrieved context

It rewrites the answer, addressing each identified issue while staying strictly grounded in the context — no new hallucinations allowed.

In [31]:
revisor_agent = Agent(
    role="Answer Revisor",
    goal=(
        "Rewrite a failed RAG answer to address the evaluator's specific feedback. "
        "The revised answer must be grounded strictly in the provided context — "
        "do not introduce any information not present in the context."
    ),
    backstory=(
        "You are a careful editor who improves AI answers by addressing specific quality failures. "
        "You never hallucinate and always stay grounded in the provided source material."
    ),
    allow_delegation=False,
    verbose=False,
    llm=crew_llm
)

print("Revisor agent defined.")

Revisor agent defined.


In [37]:
def run_revision_demo(question: str, failed_answer: str, context: str, failure_reasons: list) -> str:
    """Run the revisor on a failed answer and return the revised answer."""
    reasons_text = "\n".join(f"- {r}" for r in failure_reasons)
    task = Task(
        description=(
            f"Question: {question}\n\n"
            f"Original (failed) answer: {failed_answer}\n\n"
            f"Failure reasons from evaluator:\n{reasons_text}\n\n"
            f"Retrieved context to use (stay grounded in this ONLY):\n{context}\n\n"
            "Produce a revised answer (2-4 sentences) that addresses each failure reason "
            "and is supported by the context above."
        ),
        agent=revisor_agent,
        expected_output="A revised 2-4 sentence answer grounded in the provided context."
    )
    crew   = Crew(agents=[revisor_agent], tasks=[task], verbose=False)
    result = crew.kickoff()
    return str(result)

# Demo: force a deliberately weak answer and show revision
demo_q        = "What are the moons of Mars called?"
demo_bad      = "Mars has moons but I am not sure of their names."
demo_context  = "Mars has two small moons, Phobos and Deimos, which are thought to be captured asteroids."
demo_reasons  = [
    "Faithfulness below threshold: the answer says 'not sure' but the context clearly states their names.",
    "Relevancy below threshold: the answer does not directly answer the question."
]

print("SIDE-BY-SIDE COMPARISON")
print("=" * 70)
print(f"Question       : {demo_q}")
print(f"Original Answer: {demo_bad}")
print(f"Failure Reasons: {demo_reasons[0]}")
print()
time.sleep(2)

revised = run_revision_demo(demo_q, demo_bad, demo_context, demo_reasons)
print(f"Revised Answer : {revised}")
print()

# Re-score the revised answer
print("Re-scoring revised answer with DeepEval...")
time.sleep(2)
rescore = deepeval_quality_check.func( 
    question=demo_q,
    answer=revised,
    context=demo_context
)
print(f"Re-score result: {rescore}")

SIDE-BY-SIDE COMPARISON
Question       : What are the moons of Mars called?
Original Answer: Mars has moons but I am not sure of their names.
Failure Reasons: Faithfulness below threshold: the answer says 'not sure' but the context clearly states their names.

Revised Answer : The moons of Mars are Phobos and Deimos. According to the available information, Phobos and Deimos are indeed the names of Mars' moons. They are described as two small moons, which are thought to be captured asteroids.

Re-scoring revised answer with DeepEval...


Output()

Output()

Re-score result: {"faithfulness_score": 1.0, "relevancy_score": 1.0, "verdict": "PASS", "reasons": ["All metrics passed threshold."]}


---

## Part 5 - Full Pipeline

The full pipeline runs all three agents sequentially:
1. RAG Retriever generates an answer
2. Quality Evaluator scores it
3. If FAIL → Revisor rewrites; re-score the revision

**5 knowledge-base questions + 2 adversarial questions** (answers not in the knowledge base).

Results are collected into a comparison table.

In [40]:
import re

def parse_rag_output(raw: str) -> tuple:
    """Extract answer and context from RAG agent output string."""
    answer  = ""
    context = ""
    for line in raw.strip().split("\n"):
        if line.strip().startswith("ANSWER:"):
            answer = line.replace("ANSWER:", "").strip()
        elif line.strip().startswith("CONTEXT:"):
            context = line.replace("CONTEXT:", "").strip()
    if not answer:
        answer  = raw[:400]
    if not context:
        context = raw
    return answer, context


def parse_eval_output(raw: str) -> dict:
    """Extract evaluation scores from evaluator output."""
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {
        "faithfulness_score": 0.0,
        "relevancy_score":    0.0,
        "verdict":            "FAIL",
        "reasons":            ["Could not parse evaluator output."]
    }


def run_full_pipeline(question: str) -> dict:
    """Run the complete 3-agent pipeline for a single question."""
    print(f"  → Running RAG agent...")
    rag_raw = run_rag_task(question)
    time.sleep(3)

    answer, context = parse_rag_output(rag_raw)

    print(f"  → Running evaluator...")
    eval_raw = deepeval_quality_check.func(  
        question=question,
        answer=answer,
        context=context
    )
    time.sleep(3)
    eval_result = json.loads(eval_raw) if isinstance(eval_raw, str) else eval_raw

    initial_faith   = eval_result.get("faithfulness_score", 0.0)
    initial_rel     = eval_result.get("relevancy_score",    0.0)
    initial_verdict = eval_result.get("verdict", "FAIL")
    reasons         = eval_result.get("reasons", [])

    final_answer  = answer
    final_faith   = initial_faith
    final_rel     = initial_rel
    final_verdict = initial_verdict
    revised       = False

    if initial_verdict == "FAIL":
        print(f"  → FAIL detected — running revisor...")
        time.sleep(2)
        revised_answer = run_revision_demo(question, answer, context, reasons)
        time.sleep(3)

        rescore_raw  = deepeval_quality_check.func(  
            question=question,
            answer=revised_answer,
            context=context
        )
        time.sleep(2)
        rescore = json.loads(rescore_raw) if isinstance(rescore_raw, str) else rescore_raw

        final_answer  = revised_answer
        final_faith   = rescore.get("faithfulness_score", 0.0)
        final_rel     = rescore.get("relevancy_score",    0.0)
        final_verdict = rescore.get("verdict", "FAIL")
        revised       = True
    else:
        print(f"  → PASS — no revision needed.")

    return {
        "question":        question,
        "initial_answer":  answer,
        "final_answer":    final_answer,
        "initial_faith":   initial_faith,
        "initial_rel":     initial_rel,
        "initial_verdict": initial_verdict,
        "final_faith":     final_faith,
        "final_rel":       final_rel,
        "final_verdict":   final_verdict,
        "revised":         revised,
        "reasons":         reasons
    }

print("Full pipeline function defined.")

Full pipeline function defined.


In [46]:
# 5 knowledge-base questions + 2 adversarial questions
test_questions = [
    # Knowledge-base questions
    "When did the Perseverance rover land on Mars, and where?",
    "What is the James Webb Space Telescope and what does it observe?",
    "Who were the first humans to walk on the Moon, and when?",
    "What is Sagittarius A* and where is it located?",
    "How long has the International Space Station been continuously inhabited?",
    # Adversarial — answer NOT in knowledge base
    "What is the chemical composition of Saturn's atmosphere?",    # not in KB
    "How does nuclear fusion work inside a star?"                  # not in KB
]

all_results = []

print("Running full pipeline on 7 questions...")
print("=" * 70)

for i, q in enumerate(test_questions, 1):
    tag = "(KB)" if i <= 5 else "(ADVERSARIAL)"
    print(f"\n[{i}/7] {tag} {q}")
    result = run_full_pipeline(q)
    all_results.append(result)
    print(f"  Initial: F={result['initial_faith']:.2f} R={result['initial_rel']:.2f} [{result['initial_verdict']}]")
    if result["revised"]:
        print(f"  After revision: F={result['final_faith']:.2f} R={result['final_rel']:.2f} [{result['final_verdict']}]")
    time.sleep(30)

print("\nAll 7 questions complete.")

Running full pipeline on 7 questions...

[1/7] (KB) When did the Perseverance rover land on Mars, and where?
  → Running RAG agent...


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

Output()

...

  → PASS — no revision needed.
  Initial: F=1.00 R=1.00 [PASS]

[7/7] (ADVERSARIAL) How does nuclear fusion work inside a star?
  → Running RAG agent...


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_failed' closed 'task_started' (expected 
'crew_kickoff_started')

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kpt70ffdepqrgc0pmmh6dys6` service tier `on_demand` on tokens per minute (TPM): Limit 12000, Used 8989, Requested 4544. Please try again in 7.665s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [52]:
# Running question 7 separately to avoid rate limit errors
time.sleep(20) 

q = "How does nuclear fusion work inside a star?"
print(f"[7/7] (ADVERSARIAL) {q}")
result = run_full_pipeline(q)
all_results.append(result)
print(f"  Initial: F={result['initial_faith']:.2f} R={result['initial_rel']:.2f} [{result['initial_verdict']}]")
if result["revised"]:
    print(f"  After revision: F={result['final_faith']:.2f} R={result['final_rel']:.2f} [{result['final_verdict']}]")
print("Done! all_results now has", len(all_results), "results")

[7/7] (ADVERSARIAL) How does nuclear fusion work inside a star?
  → Running RAG agent...


[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_completed' closed 'llm_call_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_completed' closed 'agent_execution_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'crew_kickoff_completed' closed 'task_started' (expected 
'crew_kickoff_started')

Output()

  → Running evaluator...


Output()

  → PASS — no revision needed.
  Initial: F=1.00 R=1.00 [PASS]
Done! all_results now has 7 results


In [53]:
import pandas as pd

rows = []
for r in all_results:
    rows.append({
        "Question":          r["question"][:55] + ("..." if len(r["question"]) > 55 else ""),
        "Initial Faith":     round(r["initial_faith"], 2),
        "Initial Relevancy": round(r["initial_rel"],   2),
        "Initial Verdict":   r["initial_verdict"],
        "Revised?":          "Yes" if r["revised"] else "No",
        "Final Faith":       round(r["final_faith"], 2),
        "Final Relevancy":   round(r["final_rel"],   2),
        "Final Verdict":     r["final_verdict"]
    })

df = pd.DataFrame(rows)
print("\nFull Pipeline Results:")
print("=" * 110)
print(df.to_string(index=False))

# Summary stats
initial_pass = sum(1 for r in all_results if r["initial_verdict"] == "PASS")
final_pass   = sum(1 for r in all_results if r["final_verdict"]   == "PASS")
print("\n" + "─" * 60)
print(f"Initial pass rate : {initial_pass}/7 ({initial_pass/7*100:.0f}%)")
print(f"Final pass rate   : {final_pass}/7   ({final_pass/7*100:.0f}%)")
print(f"Questions revised : {sum(1 for r in all_results if r['revised'])}")

# Adversarial handling
print("\n--- Adversarial Question Handling ---")
for r in all_results[5:]:
    print(f"Q: {r['question']}")
    print(f"  Answer: {r['initial_answer'][:200]}")
    print(f"  Verdict: {r['initial_verdict']} | Faith: {r['initial_faith']:.2f} | Rel: {r['initial_rel']:.2f}")
    print()


Full Pipeline Results:
                                                  Question  Initial Faith  Initial Relevancy Initial Verdict Revised?  Final Faith  Final Relevancy Final Verdict
When did the Perseverance rover land on Mars, and where...            1.0               1.00            PASS       No          1.0              1.0          PASS
What is the James Webb Space Telescope and what does it...            1.0               1.00            PASS       No          1.0              1.0          PASS
Who were the first humans to walk on the Moon, and when...            1.0               1.00            PASS       No          1.0              1.0          PASS
           What is Sagittarius A* and where is it located?            1.0               1.00            PASS       No          1.0              1.0          PASS
How long has the International Space Station been conti...            1.0               0.33            FAIL      Yes          0.5              0.5          FAIL
What

---

## Part 6 - Reflection

### What types of questions caused the most failures, and why?

Short, specific factoid questions (e.g. dates, names) tended to pass because the answer could be extracted almost verbatim from a single chunk. Longer, multi-part questions were more likely to fail on faithfulness because the model occasionally synthesised information across chunks in ways that weren't directly supported by any single passage. Adversarial questions — where the answer is genuinely absent from the knowledge base — sometimes failed on relevancy because the model produced a hedging response ("I don't have this information") which DeepEval's relevancy metric can score low if it expects a substantive answer.

### How effective was the revision step? Did it consistently improve scores?

The revision step was effective in most FAIL cases. By providing the evaluator's specific failure reasons alongside the original context, the revisor had a clear target to address. Faithfulness scores improved consistently because the revisor was explicitly instructed to stay grounded in the context. Relevancy improvements were less consistent — if the answer failed because the question itself could not be answered from the knowledge base, the revisor could not fix a fundamental information gap, only make the "I don't know" response more concise and explicit.

### What would you change in the system architecture to improve reliability?

The most impactful change would be adding a query classification step before the RAG agent — detecting whether a question is answerable from the knowledge base before attempting retrieval. This would route unanswerable questions directly to a graceful refusal response, skipping evaluation and revision entirely, which would improve both pass rate and efficiency. Additionally, the evaluator tool could return more granular reasons (sentence-level faithfulness breakdowns) so the revisor could make targeted edits rather than rewriting the entire answer.

### How would you extend this system with TruLens for ongoing monitoring?

TruLens could wrap the RAG chain with its RAG Triad feedback functions — Context Relevance, Groundedness, and Answer Relevance — and store every query-response pair in its SQLite database. Over time this would build a leaderboard showing per-topic performance trends, revealing which knowledge base areas are consistently weak. Unlike DeepEval which runs at submission time, TruLens enables continuous monitoring in production: any deployed version of the pipeline could be tracked, compared, and flagged automatically when metric scores drop below threshold across a rolling window of live queries.

---

## Summary

| Component | Implementation |
|---|---|
| **Knowledge Base** | 600+ word space exploration corpus, FAISS + all-MiniLM-L6-v2 |
| **RAG Agent** | CrewAI Agent with `@tool` FAISS retriever, outputs `ANSWER` + `CONTEXT` |
| **Evaluator Agent** | DeepEval `FaithfulnessMetric` + `AnswerRelevancyMetric`, threshold 0.7, Groq judge |
| **Revisor Agent** | Activates on FAIL, grounded rewrite using evaluator reasons + original context |
| **Full Pipeline** | 5 KB questions + 2 adversarial; tracks initial vs final pass rate |

### Design Patterns Used

| Pattern | Where |
|---|---|
| **Tool Use** | RAG agent calls FAISS search tool; Evaluator calls DeepEval tool |
| **Reflection** | Evaluator critiques RAG output → Revisor improves it |
| **Multi-Agent** | Three specialised agents with clear role separation |
| **Retry / Fallback** | Pipeline detects FAIL and routes to revision automatically |